In [1]:
import xarray as xr
import numpy as np
import math
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib
matplotlib.use('Agg') # Ensures it runs on Gadi without a display
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import pandas as pd
from scipy.ndimage import gaussian_filter


In [2]:
# Define spatial subset parameters
lons = np.arange(80, 180)
lats = np.arange(-45, 10)

# Loading all data from NCI paths
path_u = '/g/data/rt52/era5/pressure-levels/monthly-averaged/u/**/*.nc'
path_v = '/g/data/rt52/era5/pressure-levels/monthly-averaged/v/**/*.nc'
path_t2m = '/g/data/rt52/era5/single-levels/monthly-averaged/2t/**/*.nc'
path_vimd = '/g/data/rt52/era5/single-levels/monthly-averaged/vimd/**/*.nc'
path_e = '/g/data/rt52/era5/single-levels/monthly-averaged/e/**/*.nc'
path_sst = '/g/data/m35/qr1706/PhD-1_CMIP_analysis/data uploaded/ersstv5.nc'

In [10]:
import glob

# Nouvelles coordonnées pour le vent (1 degré)
lons_wind = np.arange(80, 180, 2)
lats_wind = np.arange(-45, 10, 2)

def preprocess_wind_1deg(ds):
    if 'expver' in ds.dims:
        ds = ds.sel(expver=1)
    # On sélectionne à 1° pour le vent
    return ds.sel(level=850, longitude=lons_wind, latitude=lats_wind, method='nearest')

# On filtre les fichiers pour éviter de scanner inutilement des dossiers
# ERA5 sur Gadi est structuré par année : /g/data/rt52/era5/pressure-levels/monthly-averaged/u/YYYY/*.nc
files_u = []
files_v = []
for year in range(1940, 2024):
    files_u.extend(glob.glob(f'/g/data/rt52/era5/pressure-levels/monthly-averaged/u/{year}/*.nc'))
    files_v.extend(glob.glob(f'/g/data/rt52/era5/pressure-levels/monthly-averaged/v/{year}/*.nc'))

# Chargement de U et V avec cette nouvelle résolution
ds_u = xr.open_mfdataset(files_u, preprocess=preprocess_wind_1deg, chunks={'time': 12})
ds_v = xr.open_mfdataset(files_v, preprocess=preprocess_wind_1deg, chunks={'time': 12})
u = ds_u.u.load()
v = ds_v.v.load()
ws = (u ** 2 + v ** 2) ** 0.5

In [11]:
import glob

# 1. Définition des coordonnées (Pas de 2°)
lons = np.arange(80, 180)
lats = np.arange(-45, 10)

# 2. Fonctions de pré-traitement (pour ne charger que la zone d'étude)
def preprocess_base(ds):
    # Gère le cas où 'expver' est présent (données ERA5 récentes)
    if 'expver' in ds.dims:
        ds = ds.sel(expver=1)
    return ds.sel(longitude=lons, latitude=lats, method='nearest')

# 3. Préparation des listes de fichiers (1940-2023)
years = range(1940, 2024)
files_t2m  = []
files_vimd = []
files_ev   = []

for y in years:
    files_t2m.extend(glob.glob(f'/g/data/rt52/era5/single-levels/monthly-averaged/2t/{y}/*.nc'))
    files_vimd.extend(glob.glob(f'/g/data/rt52/era5/single-levels/monthly-averaged/vimd/{y}/*.nc'))
    files_ev.extend(glob.glob(f'/g/data/rt52/era5/single-levels/monthly-averaged/e/{y}/*.nc'))

# 4. Chargement effectif (Utilisation de chunks pour économiser la RAM)
# On utilise preprocess_base pour T2M, VIMD et EV
temp = xr.open_mfdataset(files_t2m,  preprocess=preprocess_base, chunks={'time': 12}).t2m.load()
vimd = xr.open_mfdataset(files_vimd, preprocess=preprocess_base, chunks={'time': 12}).vimd.load()
ev   = xr.open_mfdataset(files_ev,   preprocess=preprocess_base, chunks={'time': 12}).e.load()
ev_mm = ev * -1000

# 5. Chargement SST (Le fichier est déjà petit et local, open_dataset suffit)
path_sst = '/g/data/m35/qr1706/PhD-1_CMIP_analysis/data uploaded/ersstv5.nc'
dataSST = xr.open_dataset(path_sst)
sst = dataSST.sst.sel(time=slice("1940-01-01", "2023-08-01")).load()

In [12]:
# The configuration of the years remains the same
config = {
    'yr1': [
        {'years': ['1954', '1970', '1973', '1983', '1998', '2007', '2010', '2016', '2020'], 'months': range(6, 13)},
        {'years': ['1955', '1971', '1974', '1984', '1999', '2008', '2011', '2017', '2021'], 'months': range(1, 6)}
    ],
    'yr2': [
        {'years': ['1955', '1971', '1974', '1984', '1999', '2008', '2011', '2017', '2021'], 'months': range(6, 13)},
        {'years': ['1956', '1972', '1975', '1985', '2000', '2009', '2012', '2018', '2022'], 'months': range(1, 6)}
    ],
    'yr3': [
        {'years': ['1975','2000','2022'], 'months': range(6, 13)},
        {'years': ['1976','2001','2023'], 'months': range(1, 6)}
    ]
}

def calculate_clim_da(da, config):
    """Calculates monthly averages for a specific DataArray."""
    final_results = {}
    month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 
                   7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
    
    for period, sub_groups in config.items():
        all_subsets = []
        for group in sub_groups:
            years = [int(y) for y in group['years']]
            months = group['months']
            
            # Precise selection on the DataArray
            subset = da.sel(time=da.time.dt.year.isin(years) & da.time.dt.month.isin(months))
            all_subsets.append(subset)
            
        combined = xr.concat(all_subsets, dim='time')
        # Calculate the monthly average for this DataArray
        monthly_means = combined.groupby('time.month').mean(dim='time').compute()
        
        for m in monthly_means.month.values:
            m_label = f"{month_names[int(m)]}_{period}"
            final_results[m_label] = monthly_means.sel(month=m)
            
    return final_results

In [13]:
# 1. Calculation of reference climatologies (1940-2024) for each variable
clim_ev = ev_mm.groupby('time.month').mean(dim='time').compute()
clim_u    = u.groupby('time.month').mean(dim='time').compute()
clim_v    = v.groupby('time.month').mean(dim='time').compute()
clim_ws   = ws.groupby('time.month').mean(dim='time').compute()
clim_temp = temp.groupby('time.month').mean(dim='time').compute()
clim_sst  = sst.groupby('time.month').mean(dim='time').compute()
clim_vimd = vimd.groupby('time.month').mean(dim='time').compute()

# 2. Calculation of averages per period (Yr1, Yr2, Yr3)
res_ev = calculate_clim_da(ev_mm, config)
res_u    = calculate_clim_da(u, config)
res_v    = calculate_clim_da(v, config)
res_ws   = calculate_clim_da(ws, config)
res_temp = calculate_clim_da(temp, config)
res_sst  = calculate_clim_da(sst, config)
res_vimd = calculate_clim_da(vimd, config)

# 3. Calculating anomalies
def get_anomalies(res_dict, clim_da):
    anom_dict = {}
    month_map = {"Jan":1, "Feb":2, "Mar":3, "Apr":4, "May":5, "Jun":6, 
                 "Jul":7, "Aug":8, "Sep":9, "Oct":10, "Nov":11, "Dec":12}
    
    for key, data in res_dict.items():
        m_name = key.split('_')[0]
        m_idx = month_map[m_name]
        anom_dict[key] = data - clim_da.sel(month=m_idx)
    return anom_dict

anom_ev = get_anomalies(res_ev, clim_ev)
anom_u    = get_anomalies(res_u, clim_u)
anom_v    = get_anomalies(res_v, clim_v)
anom_ws   = get_anomalies(res_ws, clim_ws)
anom_temp = get_anomalies(res_temp, clim_temp)
anom_sst  = get_anomalies(res_sst, clim_sst)
anom_vimd = get_anomalies(res_vimd, clim_vimd)


In [14]:
# Assure-toi que cette fonction est bien définie ainsi :
def get_wind_plot_data(season_tag, skip=1, threshold=0.75):
    u_da = seasonal_u[season_tag]
    v_da = seasonal_v[season_tag]
    anom_speed = np.sqrt(u_da**2 + v_da**2)
    
    u_f = u_da.where(anom_speed >= threshold)
    v_f = v_da.where(anom_speed >= threshold)
    
    # Si skip=1 et que u_da est à 1°, on aura 1 flèche par degré
    if skip > 1:
        u_plot = u_f.thin({'latitude': skip, 'longitude': skip})
        v_plot = v_f.thin({'latitude': skip, 'longitude': skip})
    else:
        u_plot = u_f
        v_plot = v_f
        
    return u_plot, v_plot

def get_seasonal_anomaly_da(anom_dict, months_list, period_suffix):
    keys = [f"{m}_{period_suffix}" for m in months_list]
    da_list = [anom_dict[k] for k in keys if k in anom_dict]
    
    if not da_list:
        return None
        
    return xr.concat(da_list, dim='month_axis').mean(dim='month_axis').compute()

# --- Seasons CONFIGURATION ---
seasons_config = {
    'DJF': ['Dec', 'Jan', 'Feb'],
    'SON': ['Sep', 'Oct', 'Nov'],
    'JJA': ['Jun', 'Jul', 'Aug']
}

# --- INITIALISATION OF DICTIONARIES ---
seasonal_u = {}
seasonal_v = {}
seasonal_ws = {}
seasonal_vimd = {}
seasonal_temp = {}
seasonal_sst = {}
seasonal_ev = {}

# 1. Calculation of periods yr1 and yr3 for each season and each variable
for s_name, m_list in seasons_config.items():
    for p in ['yr1', 'yr3']:
        tag = f'{s_name}_{p}'
        seasonal_u[tag]    = get_seasonal_anomaly_da(anom_u, m_list, p)
        seasonal_v[tag]    = get_seasonal_anomaly_da(anom_v, m_list, p)
        seasonal_ws[tag]   = get_seasonal_anomaly_da(anom_ws, m_list, p)
        seasonal_vimd[tag] = get_seasonal_anomaly_da(anom_vimd, m_list, p)
        seasonal_temp[tag] = get_seasonal_anomaly_da(anom_temp, m_list, p)
        seasonal_sst[tag]  = get_seasonal_anomaly_da(anom_sst, m_list, p)
        seasonal_ev[tag] = get_seasonal_anomaly_da(anom_ev, m_list, p)

# 2. Calculation of differences (Yr3 - Yr1)
for s_name in seasons_config.keys():
    diff_tag = f'{s_name}_diff'
    y3, y1 = f'{s_name}_yr3', f'{s_name}_yr1'
    
    if y3 in seasonal_u and y1 in seasonal_u:
        seasonal_u[diff_tag]    = seasonal_u[y3] - seasonal_u[y1]
        seasonal_v[diff_tag]    = seasonal_v[y3] - seasonal_v[y1]
        seasonal_ws[diff_tag]   = seasonal_ws[y3] - seasonal_ws[y1]
        seasonal_vimd[diff_tag] = seasonal_vimd[y3] - seasonal_vimd[y1]
        seasonal_temp[diff_tag] = seasonal_temp[y3] - seasonal_temp[y1]
        seasonal_sst[diff_tag]  = seasonal_sst[y3] - seasonal_sst[y1]
        seasonal_ev[diff_tag]   = seasonal_ev[y3] - seasonal_ev[y1]

In [15]:
# --- 1. PARAMÈTRES ET NIVEAUX ---
seasons_list = ['SON', 'DJF']
periods_list = ['yr1', 'yr3', 'diff']
row_titles = ["Year 1", "Year 3", "Year 3 - Year 1"]
abc_labels = [['a)', 'b)', 'c)', 'd)'], 
              ['e)', 'f)', 'g)', 'h)'], 
              ['i)', 'j)', 'k)', 'l)']]

lvl_temp = [-1.5, -1.3, -1.1, -0.9, -0.7, -0.5, -0.3, -0.1, 
             0.1,  0.3,  0.5,  0.7,  0.9,  1.1,  1.3,  1.5]
cmap_temp = plt.get_cmap('bwr')
norm_temp = mcolors.BoundaryNorm(boundaries=lvl_temp, ncolors=cmap_temp.N, extend='both')

lvl_vimd = np.arange(-4, 4.5, 0.5) 

lev_pos = [0.2, 0.4, 0.6, 0.8,1]
lev_neg = [-0.2, -0.4, -0.6, -0.8,-1]

n_values_list = ['9', '3', None]


In [16]:
# --- 2. CONSTRUCTION DE LA FIGURE ---
fig, axes = plt.subplots(3, 4, figsize=(26, 16), subplot_kw={'projection': ccrs.PlateCarree()})
plt.subplots_adjust(wspace=0.1, hspace=-0.48, left=0.1)

for r, period in enumerate(periods_list):
    axes[r, 0].annotate(row_titles[r], xy=(-0.11, 0.5), xycoords='axes fraction',
                        fontsize=18, ha='center', va='center', rotation=90)
    
    current_n = n_values_list[r]
    
    for c_seas, SEAS in enumerate(seasons_list):
        tag = f"{SEAS}_{period}"
        col_offset = c_seas * 2
        
        for sub_col in range(2):
            ax = axes[r, col_offset + sub_col]
            label = abc_labels[r][col_offset + sub_col]
            
            ax.set_extent([90, 170, -42.4, 0], crs=ccrs.PlateCarree())
            ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color='black', zorder=6)
            
            ax.text(0.02, 0.9, label, transform=ax.transAxes, fontsize=16,
                    bbox=dict(facecolor='white', edgecolor='none', alpha=1), zorder=10)
            
            if current_n is not None:
                ax.text(0.02, 0.04, f"n={current_n}", transform=ax.transAxes, fontsize=14,
                        bbox=dict(facecolor='white', edgecolor='none', alpha=1), zorder=10)
            
            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.3, linestyle='--')
            gl.top_labels = gl.right_labels = False
            gl.xlabel_style = gl.ylabel_style = {'size': 8}
            
        ax_l = axes[r, col_offset]
        ax_r = axes[r, col_offset + 1]

        # --- GAUCHE : TEMP / SST / EVAP ---
        shading_kw = {'levels': lvl_temp, 'norm': norm_temp, 'cmap': 'bwr', 
                      'add_colorbar': False, 'add_labels': False, 'extend': 'both', 'linewidths': 0}
        
        im_temp = seasonal_temp[tag].plot.contourf(ax=ax_l, **shading_kw)
        seasonal_sst[tag].plot.contourf(ax=ax_l, **shading_kw)

        ev_raw = seasonal_ev[tag]
        ev_smooth = gaussian_filter(ev_raw.values, sigma=1.5)
        lons_ev, lats_ev = ev_raw.longitude, ev_raw.latitude
        common_ev = {'transform': ccrs.PlateCarree(), 'zorder': 7}
        
        for lv, ls, lw in zip(lev_pos, ['--', '-', '-', '-','-'], [1, 1, 1.5, 2,2.5]):
            c = ax_l.contour(lons_ev, lats_ev, ev_smooth, levels=[lv], colors='red', linestyles=ls, linewidths=lw, **common_ev)
            ax_l.clabel(c, inline=True, fontsize=8, fmt='%1.1f')
            
        for lv, ls, lw in zip(lev_neg, ['--', '-', '-', '-','-'], [1, 1, 1.5, 2,2.5]):
            c = ax_l.contour(lons_ev, lats_ev, ev_smooth, levels=[lv], colors='blue', linestyles=ls, linewidths=lw, **common_ev)
            ax_l.clabel(c, inline=True, fontsize=8, fmt='%1.1f')

        # --- DROITE : VIMD / VENT ---
        im_vimd = seasonal_vimd[tag].plot.contourf(
            ax=ax_r, levels=lvl_vimd, cmap='PuOr_r', 
            add_colorbar=False, add_labels=False, extend='both', linewidths=0
        )
        
        # Dans la boucle de plot, section DROITE : VIMD / VENT
        u_p, v_p = get_wind_plot_data(tag, skip=1, threshold=0.75)
        Q = ax_r.quiver(u_p.longitude, u_p.latitude, u_p, v_p, 
                        scale=40, width=0.002, color='black', 
                        transform=ccrs.PlateCarree(), zorder=8)

# --- 3. ÉLÉMENTS FIXES ---
axes[0, 0].set_title("SON: Temp & Evap", fontsize=18, pad=10)
axes[0, 1].set_title("SON: VIMD & Wind", fontsize=18, pad=10)
axes[0, 2].set_title("DJF: Temp & Evap", fontsize=18, pad=10)
axes[0, 3].set_title("DJF: VIMD & Wind", fontsize=18, pad=10)

cax1 = fig.add_axes([0.91, 0.52, 0.012, 0.25])
cb1 = fig.colorbar(im_temp, cax=cax1)
cb1.set_ticks([-1.5, -1, -0.5, 0, 0.5, 1, 1.5])
cb1.set_label('Temp/SST Anom. (K)', size=14)

cax2 = fig.add_axes([0.91, 0.24, 0.012, 0.25])
cb2 = fig.colorbar(im_vimd, cax=cax2)
cb2.set_label('VIMD Anom. (kg/m$^2$)', size=14)

axes[2, 3].quiverkey(Q, X=1.09, Y=0.04, U=1, label='1 m/s', labelpos='S', coordinates='axes', fontproperties={'size': 12})

# Saves directly to your current working directory on Gadi
plt.savefig('/g/data/m35/qr1706/PhD-1_ERA5_plot/Plots & graphs/Fours-y-tout_SON_DJF_ERA5.png', bbox_inches='tight', dpi=300)
plt.show()

/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1655: UserWarning: linewidths is ignored by contourf
  result = super().contourf(*args, **kwargs)
/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1655: UserWarning: linewidths is ignored by contourf
  result = super().contourf(*args, **kwargs)
/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1604: UserWarning: No contour levels were found within the data range.
  result = super().contour(*args, **kwargs)
/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1604: UserWarning: No contour levels were found within the data range.
  result = super().contour(*args, **kwargs)
/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1604: UserWarning: No contour levels were found within the data range.
  result = super().contour(*args, **kwargs)
/home/561/qr1706/.local/lib/python3.9/site-packages/cartopy/mpl/geoaxes.py:1604: Use

NameError: name 'seasonal_mmm' is not defined